# 12 — Symptom severity: Cam-CAN vs HCP

How much clinical signal is there to predict, in each cohort? Three figures and two tables.

**Read this before the numbers.** The two cohorts are measured with different instruments, whose
published cutoffs were drawn for different purposes:

| | instrument | range | bands | what the first cutoff means |
|---|---|---|---|---|
| Cam-CAN | HADS, per subscale | 0–21 | Normal / Mild / Moderate / Severe at 0–7 / 8–10 / 11–14 / 15–21 | a **screening** line, chosen for sensitivity |
| HCP | ASR, DSM-oriented | raw, reported as T | Normal / Borderline / Clinical at T<65 / 65–69 / ≥70 | the **93rd percentile** of a reference sample, by construction |

So `% mild-or-worse` in Cam-CAN and `% borderline-or-worse` in HCP are **not comparable**. A gap
between them is mostly a gap between two conventions. The comparison that *is* valid lives in F3
and in the percentile table: within-cohort percentiles, the floor fraction, and the spread.

**Bands are described on T; the model is fitted on raw.** A T-score has age and sex normed out of
it in the reference sample, which HADS never had. Regressing a T-score against brain states would
quietly remove age and sex from one cohort and not the other — exactly the dimension this design
is about.

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

import pandas as pd

from fmri_decomposition.supervised_model import symptoms as sd   # bands, id matching, load()
from fmri_decomposition.figures import symptoms as sp            # the three figures

# ---- point these at your own copies -----------------------------------------
HCP_PHENO    = Path("/project/6008063/tamires/cohorts/hcp_pheno/participants_pheno.csv")
HCP_SUBJECTS = REPO / "config" / "hcp7t_fix_participants.csv"
CAMCAN_DIR   = Path("/project/6008063/tamires/cohorts/camcan/dataman/useraccess/opendata/"
                    "paule_toussaint_camcan01870")
# Several tables are unioned. Cam-CAN's movie and rest runs were discovered as
# two cohorts before they were merged into one, so the subjects with imaging are
# the union of the two older tables. If you have rebuilt the merged
# config/camcan_participants.csv, a single path works just as well.
CAMCAN_SUBJECTS = [REPO / "config" / "camcan_movie_participants.csv",
                   REPO / "config" / "camcan_rest_participants.csv"]
SAVE_DIR     = None        # e.g. REPO / "notebooks" / "figures" / "12_symptoms"
# -----------------------------------------------------------------------------

d = sd.load(HCP_PHENO, HCP_SUBJECTS, CAMCAN_DIR, CAMCAN_SUBJECTS)
for coh, blk in d.items():
    print(f"{coh:7s} n={blk['n']:4d}  {blk['instrument']:18s} scales={list(blk['scales'])}")
    if blk["missing_pheno"]:
        print(f"         !! {len(blk['missing_pheno'])} imaging subjects with no phenotype row")

## Data quality first

Two things that have to be checked before any number below is believed.

**Impossible values.** Cam-CAN's `additional_HADS_depression` carries **9999** for "no answer".
HADS runs 0–21, so that is a missing-data sentinel, not a score. Left in, it sets the max to 9999,
inflates the sd by two orders of magnitude, squashes the histogram into the first pixel column, and
— if the column is used as a modelling target — hands the regression a few enormous outliers to
chase. Anything outside `[0, 21]` is therefore dropped from the HADS columns, and what was dropped
is printed, because a silent exclusion is its own bug.

The ASR raw scores have no published ceiling, so nothing is dropped there; a value that *looks*
like a sentinel (a run of 9s, or negative) is reported as **suspect** and kept, and you decide.

In [ ]:
drops = []
for coh, blk in d.items():
    for con, sc in blk["scales"].items():
        dr = sc.get("dropped") or {}
        if dr.get("n_dropped") or dr.get("n_suspect"):
            drops.append({"cohort": coh, "construct": con, **dr})
if drops:
    display(pd.DataFrame(drops))
else:
    print("no impossible or suspect value in any symptom column")

## Table 1 — bands, absolute and percent

`Zero` is split out of the lowest band. That split needs no per-subject join: a raw score of 0 is
below every cutoff on every scale here, so a zero-scoring subject is necessarily in the lowest band.

In [ ]:
bands = pd.DataFrame(sp.band_table(d))
wide = (bands.assign(cell=lambda x: x["n"].astype(str) + "  (" + x["pct"].round(1).astype(str) + "%)")
             .pivot(index=["cohort", "construct"], columns="band", values="cell")
             .fillna("—"))
display(wide)
note = bands["split_note"].iloc[0]
print(note if note else "zero was split out of the lowest band for every row")

## Table 2 — the comparable numbers

Percentiles within each cohort, the fraction sitting at the floor, and the spread. `sd_over_max` is
blank for the ASR because that scale has no published maximum here.

**`pct_at_floor` is the number that decides whether a target is usable.** Spearman over a column
where a large share of subjects tie at zero has far less power than its *n* suggests, and more
subjects do not fix a target that is mostly floor.

In [ ]:
summ = pd.DataFrame(sp.summary_table(d))
display(summ)

## F1 — the raw distributions, each on its own axis

Separate axes on purpose. Dashed rules are the HADS cutoffs; the ASR panels carry none, because the
ASR's cutoffs are defined on T and have no position on a raw axis.

In [ ]:
_ = sp.fig_histograms(d, save_dir=SAVE_DIR)

## F2 — band composition

The figure a reader will screenshot without the notebook around it, so the caveat is printed on it.

In [ ]:
_ = sp.fig_bands(d, save_dir=SAVE_DIR)

## F3 — Q-Q: the honest side-by-side

One point per percentile: its *x* is the score Cam-CAN reaches there, its *y* what HCP reaches. No
shared axis and no assumed maximum, so this is the comparison the band percentages cannot make.

How to read it: a **flat run** along either axis is that cohort's floor. A curve that **rises
steeply** means HCP's scale spreads faster than Cam-CAN's at those percentiles; **shallow** means the
opposite. The line is neutral grey, not a cohort colour, because it is the relationship between the
two — not either one of them.

In [ ]:
_ = sp.fig_quantiles(d, save_dir=SAVE_DIR)

## Every other HCP scale

`DSM_Anxi` and `DSM_Depr` lead because they pair with the two HADS subscales. But the battery
carries ~27 scales in four families, and the pairing with HADS is not the only thing that matters:

| family | what it is |
|---|---|
| **DSM-oriented** | built to line up with DSM categories — this is what pairs with a HADS subscale |
| **syndrome** | the empirically derived ASR syndromes. Narrower, and `Anxd` deliberately mixes anxiety with depression |
| **broad-band** | sums over several syndromes. **More items ⇒ lower floor and more resolution**, often the better regression target even with no HADS counterpart |
| **SSAGA** | diagnostic-interview counts and flags. A different kind of measurement — some are 0/1, where Spearman is a point-biserial and the "floor" is just a base rate |

The ranking below is by **floor fraction**, the share of subjects sitting at the scale minimum. That
is what decides power: a Spearman over a column where most subjects tie at the floor has far less
power than its *n* suggests, and more subjects do not fix it — which is the whole reason this
cohort's *n* was raised in the first place.

Raw scores throughout, never T, for the reason in the first cell.

In [ ]:
scales = sd.hcp_scales(HCP_PHENO, HCP_SUBJECTS)
hcp = pd.DataFrame(sp.hcp_scale_table(scales))
print(f"{len(hcp)} scales, {hcp['family'].nunique()} families")
# pct_at_floor = share AT THE MINIMUM. For a 0/1 flag that is the share scoring
# zero, so pct_above_floor is the positive rate -- the number a reader of that
# row actually wants.
display(hcp[["family", "label", "column", "n", "median", "p90", "max", "sd",
             "n_distinct", "binary", "pct_at_floor", "pct_above_floor",
             "pairs_with"]])

# Nothing symptom-shaped should vanish without a reason being printed.
skipped = sd.hcp_skipped(HCP_PHENO, HCP_SUBJECTS)
if skipped:
    print(f"\n{len(skipped)} symptom-looking column(s) NOT in the inventory:")
    display(pd.DataFrame(skipped))
else:
    print("\nno symptom-looking column was left out")

### F4 — which scales leave variance to predict

**Read the bar as emptiness, not as prevalence.** The axis is the share of subjects sitting at the
scale *minimum*, so a long bar means the scale is mostly unused — not that the symptom is common.

On a 0/1 flag this reads backwards from what it looks like: agoraphobia at 93% means **93% scored
zero**, i.e. **7% have it**. Every binary row therefore writes its positive rate out, and the
binaries are sorted to the bottom of their family rather than ranked beside the counts.

`N values` on the right is the number of *distinct values observed* across the 168 subjects — the
scale's resolution. `total problems` with 62 distinct values discriminates far more finely than
`intrusive` with 8. **Shorter bar and more distinct values = better target.** The count is printed
rather than plotted: it is a different measure on a different scale, and giving it a second x-axis
would be a dual-axis chart.

In [ ]:
_ = sp.fig_hcp_scales(scales, save_dir=SAVE_DIR)

### F5 — the shape of each one

Small multiples, each on its own axis. A scale with a tall first bar and a long thin tail is mostly
floor, whatever its median says.

In [ ]:
_ = sp.fig_hcp_grid(scales, save_dir=SAVE_DIR)

### F6 — the T-scores, with the lines that have a published meaning

The raw figures above answer *how much variance is there to predict*, which is what picks the
modelling target. This one answers a different question: *how many people are above a clinical
line* — which only T can answer, because only T has one.

Three things to hold while reading it:

- **T is not "more processed" than raw.** It is the same score re-expressed as a position in the
  ASR's reference sample, normed by age and sex.
- **T ≥ 65 is that reference's 93rd percentile by construction.** So "12% clinical" here means this
  cohort has more people up there than the reference, not that 12% are ill. And it is still not
  comparable with a HADS screening percentage, for the reason in the first cell.
- **Two threshold pairs.** The narrow scales (DSM-oriented, syndrome) use 65/70; the broad-band
  scales use 60/64. Each panel draws and names its own.

Counts are printed as *n* and % because with a cohort this healthy some panels have one or two
subjects above the line, and a bar that thin is invisible.

SSAGA does not appear: those columns have no `_T`.

In [ ]:
tscales = sd.hcp_t_scales(HCP_PHENO, HCP_SUBJECTS)
tsum = pd.DataFrame([{k: v for k, v in r.items() if k != "values"}
                     for r in tscales])
display(tsum[["family", "label", "column", "n", "median", "max",
              "borderline_t", "n_borderline_plus", "pct_borderline_plus",
              "clinical_t", "n_clinical", "pct_clinical", "pairs_with"]])
_ = sp.fig_hcp_t_grid(tscales, save_dir=SAVE_DIR)

### Where mild / moderate / severe begin, on the raw scale

**There is no published raw cutoff.** The ASR's bands are defined on the **T-score** — ≥65
borderline, ≥70 clinical (≥60 / ≥64 for the broad-band scales) — and raw → T goes through the
manual's norm tables, which are normed **by age and sex** and are not in the data. So the question
"what raw score is moderate?" has no answer from the instrument.

What *can* be done is read it off this cohort's own paired `_Raw` / `_T` columns. The result is not
a number but a **range**, and that is the point: because T is normed by age and sex, two subjects
with the same raw score can land on opposite sides of 65. Where the ranges overlap, the raw score
alone does not decide the band.

SSAGA columns have no `_T`, so they get no row — there is nothing to read the band from, published
or empirical.

This also means: **if you want bands, band on T; if you want to fit, fit on raw.** Doing both on
one column is what the overlap below makes impossible.

In [ ]:
cuts = pd.DataFrame(sd.hcp_raw_cutoffs(HCP_PHENO, HCP_SUBJECTS))
display(cuts[["family", "label", "column", "n",
              "Borderline_from", "Borderline_note",
              "Clinical_from", "Clinical_note"]])

### Picking a target

Three things to weigh, in this order:

1. **Floor fraction.** Below ~10% is comfortable; above ~30% the target is effectively a tail
   variable and the effective *n* is far below 168.
2. **Does it pair with HADS?** Only `DSM_Anxi` and `DSM_Depr` do. A target with no Cam-CAN
   counterpart can still be modelled within HCP, but it cannot enter a cross-cohort comparison.
3. **Is it binary?** The SSAGA flags are 0/1. Spearman on them is a point-biserial, and they belong
   with `--model logistic`, not with ridge or hgb.

The usual tension: the broad-band scales have the lowest floor and the most resolution but no HADS
counterpart, while the two that pair with HADS sit higher up the list. If the four-table comparison
is the goal, that argues for `DSM_Anxi` / `DSM_Depr` as primary and a broad-band scale as the
within-HCP sensitivity check — not the other way round.

## Families, for `--group-col`

HCP contains twins and siblings. With *n* = 168 this stops being a footnote: two members of a family
in different CV folds leak. The counts below say how many groups a `GroupKFold` by `Family_ID` would
actually have, and whether any cluster is larger than a pair.

In [ ]:
from collections import Counter

rows, _, _ = sd._hcp_rows(HCP_PHENO, HCP_SUBJECTS)
fam = Counter((r.get("Family_ID") or "").strip() for r in rows if (r.get("Family_ID") or "").strip())
sizes = Counter(fam.values())
print(f"subjects : {sum(fam.values())}")
print(f"families : {len(fam)}")
print(f"cluster sizes: {dict(sorted(sizes.items()))}")
print(f"-> GroupKFold by Family_ID has {len(fam)} groups; largest cluster {max(sizes)} subject(s)")